# 05. Preparació de CSV per a Flourish

Aquest notebook prepara els fitxers CSV necessaris per crear les visualitzacions finals a Flourish.

L'objectiu és transformar el dataset principal i els resultats dels notebooks anteriors en taules adaptades a cada escena de la visualització.

Es generen CSV per a les escenes 04 a 09:

- Escena 04: NAO i precipitació.
- Escena 05: NAO, i humitat del sòl.
- Escena 06: WeMO, temperatura i SST.
- Escena 07: comparació entre NAO, WeMO i RONI.
- Escena 08: augment d'episodis càlids amb baixa humitat del sòl.
- Escena 09: explorador interactiu de correlacions.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

BASE = Path("..") if Path("../data_processed").exists() else Path(".")

PROC = BASE / "data_processed"
FLOURISH = BASE / "data_flourish"

FLOURISH.mkdir(parents=True, exist_ok=True)

DATA_MAIN = PROC / "teleconnections_catalonia_1981_2020_main_v2.csv"

print("PROC:", PROC.resolve())
print("FLOURISH:", FLOURISH.resolve())

PROC: /workspace/data_processed
FLOURISH: /workspace/data_flourish


In [2]:
df = pd.read_csv(DATA_MAIN, parse_dates=["data"])

print("Dataset principal carregat:")
print(df.shape)

display(df.head())

Dataset principal carregat:
(1920, 38)


,data,any,mes,dies_mes,estacio,periode,subregio,nao,wemo,roni,...,anomalia_sst,mes_molt_calid,mes_molt_fred,mes_molt_sec,mes_molt_humit,sol_molt_sec,sol_molt_humit,episodi_calid_i_sec,episodi_fred_i_humit,episodi_calid_i_sol_sec
0,1981-01-01,1981,1,31,Winter,1981-1990,Llevant_Iberic,1.00,1.27,0.0,...,-0.891328,False,True,False,False,True,False,False,False,False
1,1981-02-01,1981,2,28,Winter,1981-1990,Llevant_Iberic,1.04,-0.18,-0.3,...,-0.814626,False,True,False,False,True,False,False,False,False
2,1981-03-01,1981,3,31,Spring,1981-1990,Llevant_Iberic,0.01,0.48,-0.3,...,-0.302593,True,False,False,False,True,False,False,False,True
3,1981-04-01,1981,4,30,Spring,1981-1990,Llevant_Iberic,-3.04,-1.36,-0.3,...,-0.168856,False,False,False,True,False,True,False,False,False
4,1981-05-01,1981,5,31,Spring,1981-1990,Llevant_Iberic,0.05,1.69,-0.3,...,-1.155891,False,True,False,False,False,True,False,False,False


In [3]:
# Comprovem el dataset

required_cols = [
    "data", "any", "mes", "estacio", "periode", "subregio",
    "nao", "wemo", "roni",
    "fase_nao", "fase_wemo", "fase_enso",
    "anomalia_temperatura",
    "anomalia_precipitacio_mensual_mm",
    "anomalia_humitat_sol_capa1",
    "anomalia_sst",
    "precipitacio_mensual_mm",
    "humitat_sol_capa1",
    "episodi_calid_i_sol_sec",
]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Falten columnes necessàries al dataset principal: {missing}")

print("Rang temporal:", df["data"].min(), "->", df["data"].max())
print("Duplicats data-subregio:", df.duplicated(subset=["data", "subregio"]).sum())
print("Files:", len(df))

Rang temporal: 1981-01-01 00:00:00 -> 2020-12-01 00:00:00
Duplicats data-subregio: 0
Files: 1920


In [4]:
# Funcions auxiliars

def export_csv(dataframe, filename, traduir=True, ordenar=True):
    """
    Exporta un dataframe a data_flourish i mostra informació bàsica.
    Per defecte tradueix i ordena valors categòrics per als CSV finals de Flourish.
    """
    out = dataframe.copy()

    if traduir:
        out = traduir_valors_flourish(out)

    if ordenar:
        out = ordenar_categories_flourish(out)

    path = FLOURISH / filename
    out.to_csv(path, index=False, encoding="utf-8-sig")

    print(f"Exportat: {path}")
    print("Dimensions:", out.shape)
    display(out.head())

    return path


def add_data_text_column(dataframe):
    """
    Afegeix una columna Data en format YYYY-MM.
    No modifica la columna original data.
    """
    out = dataframe.copy()
    out["Data"] = out["data"].dt.strftime("%Y-%m")
    return out


def select_existing_columns(dataframe, columns):
    """
    Selecciona només les columnes que existeixen.
    Evita errors si algun dataset té menys variables.
    """
    return [c for c in columns if c in dataframe.columns]

In [5]:
# Diccionaris de variables per etiquetes de Flourish

# Aquestes etiquetes només s'utilitzen en columnes noves com "Variable".
# No es modifiquen els valors originals de les columnes del dataset.

variables_narratives = {
    "anomalia_temperatura": {
        "Variable": "Temperatura",
        "Unitat": "°C",
    },
    "anomalia_precipitacio_mensual_mm": {
        "Variable": "Precipitació",
        "Unitat": "mm",
    },
    "anomalia_humitat_sol_capa1": {
        "Variable": "Humitat del sòl",
        "Unitat": "m3/m3",
    },
    "anomalia_sst": {
        "Variable": "SST",
        "Unitat": "°C",
    },
}

ordre_variables_narratives = list(variables_narratives.keys())

teleconnexio_labels = {
    "nao": "NAO",
    "wemo": "WeMO",
    "roni": "RONI",
}

print(variables_narratives)

{'anomalia_temperatura': {'Variable': 'Temperatura', 'Unitat': '°C'}, 'anomalia_precipitacio_mensual_mm': {'Variable': 'Precipitació', 'Unitat': 'mm'}, 'anomalia_humitat_sol_capa1': {'Variable': 'Humitat del sòl', 'Unitat': 'm3/m3'}, 'anomalia_sst': {'Variable': 'SST', 'Unitat': '°C'}}


In [6]:
# Ordres i traduccions per als CSV finals de Flourish.
# No modifiquem el dataset principal abans dels càlculs.

ordre_estacions_original = ["Winter", "Spring", "Summer", "Autumn"]
ordre_estacions_cat = ["Hivern", "Primavera", "Estiu", "Tardor"]

ordre_fases_original = ["Negative", "Neutral", "Positive"]
ordre_fases_cat = ["Negativa", "Neutral", "Positiva"]

ordre_enso_original = ["La Nina", "Neutral", "El Nino"]
ordre_enso_cat = ["La Niña", "Neutral", "El Niño"]

ordre_subregions_original = [
    "Catalunya_Balears",
    "Golf_Lleo",
    "Llevant_Iberic",
    "Mediterrani_Central_Occidental",
]

ordre_subregions_cat = [
    "Catalunya-Balears",
    "Golf del Lleó",
    "Llevant Ibèric",
    "Mediterrani central occidental",
]

trad_estacions = dict(zip(ordre_estacions_original, ordre_estacions_cat))
trad_fases = dict(zip(ordre_fases_original, ordre_fases_cat))
trad_enso = dict(zip(ordre_enso_original, ordre_enso_cat))
trad_subregions = dict(zip(ordre_subregions_original, ordre_subregions_cat))

trad_ambits = {
    "global": "Global",
    "estacio": "Estació",
    "subregio": "Subregió",
    "periode": "Període",
    "subregio_estacio": "Subregió i estació",
    "subregio_periode": "Subregió i període",
    "estacio_periode": "Estació i període",
    "subregio_estacio_periode": "Subregió, estació i període",
}

ordre_ambits_cat = [
    "Global",
    "Estació",
    "Subregió",
    "Subregió i estació",
    "Període",
    "Subregió i període",
    "Estació i període",
    "Subregió, estació i període",
]

def traduir_valors_flourish(dataframe):
    """
    Tradueix valors categòrics només per als CSV finals de Flourish.
    """
    out = dataframe.copy()

    for col in ["Estació", "estacio"]:
        if col in out.columns:
            out[col] = out[col].replace(trad_estacions)

    for col in ["Fase NAO", "Fase WeMO", "fase_nao", "fase_wemo"]:
        if col in out.columns:
            out[col] = out[col].replace(trad_fases)

    for col in ["Fase ENSO", "fase_enso"]:
        if col in out.columns:
            out[col] = out[col].replace(trad_enso)

    for col in ["Àmbit", "ambit"]:
        if col in out.columns:
            out[col] = out[col].replace(trad_ambits)

    for col in ["Subregió", "subregio"]:
        if col in out.columns:
            out[col] = out[col].replace(trad_subregions)

    return out


def ordenar_categories_flourish(dataframe):
    """
    Ordena les files dels CSV perquè Flourish respecti millor l'ordre visual.
    """
    out = dataframe.copy()

    if "Fase NAO" in out.columns:
        out["Fase NAO"] = pd.Categorical(out["Fase NAO"], categories=ordre_fases_cat, ordered=True)

    if "Fase WeMO" in out.columns:
        out["Fase WeMO"] = pd.Categorical(out["Fase WeMO"], categories=ordre_fases_cat, ordered=True)

    if "Fase ENSO" in out.columns:
        out["Fase ENSO"] = pd.Categorical(out["Fase ENSO"], categories=ordre_enso_cat, ordered=True)

    if "Estació" in out.columns:
        out["Estació"] = pd.Categorical(out["Estació"], categories=ordre_estacions_cat, ordered=True)

    if "Subregió" in out.columns:
        out["Subregió"] = pd.Categorical(out["Subregió"], categories=ordre_subregions_cat, ordered=True)

    if "Àmbit" in out.columns:
        out["Àmbit"] = pd.Categorical(out["Àmbit"], categories=ordre_ambits_cat, ordered=True)

    sort_cols = [
        c for c in [
            "Fase NAO",
            "Fase WeMO",
            "Fase ENSO",
            "Estació",
            "Subregió",
            "Període",
            "Teleconnexió",
            "Variable",
            "Àmbit",
        ]
        if c in out.columns
    ]

    if sort_cols:
        out = out.sort_values(sort_cols).reset_index(drop=True)

    # Tornem a string per evitar problemes d'exportació amb categories.
    for col in out.select_dtypes(include="category").columns:
        out[col] = out[col].astype(str)

    return out

## Escena 04 — NAO i precipitació hivernal

In [7]:
escena_04 = add_data_text_column(df)

escena_04["Es hivern"] = escena_04["estacio"] == "Winter"

cols_04 = [
    "Data",
    "any",
    "mes",
    "estacio",
    "subregio",
    "periode",
    "nao",
    "fase_nao",
    "anomalia_precipitacio_mensual_mm",
    "precipitacio_mensual_mm",
    "Es hivern",
]

escena_04 = escena_04[cols_04].rename(columns={
    "any": "Any",
    "mes": "Mes",
    "estacio": "Estació",
    "subregio": "Subregió",
    "periode": "Període",
    "nao": "NAO",
    "fase_nao": "Fase NAO",
    "anomalia_precipitacio_mensual_mm": "Anomalia precipitació mensual (mm)",
    "precipitacio_mensual_mm": "Precipitació mensual (mm)",
})

export_csv(
    escena_04,
    "escena_04_nao_precipitacio_hivern.csv"
)

Exportat: ../data_flourish/escena_04_nao_precipitacio_hivern.csv
Dimensions: (1920, 11)


,Data,Any,Mes,Estació,Subregió,Període,NAO,Fase NAO,Anomalia precipitació mensual (mm),Precipitació mensual (mm),Es hivern
0,1981-12,1981,12,Hivern,Catalunya-Balears,1981-1990,-2.20,Negativa,-2.000200,49.811482,True
1,1982-01,1982,1,Hivern,Catalunya-Balears,1981-1990,-0.72,Negativa,7.160294,58.351951,True
2,1983-02,1983,2,Hivern,Catalunya-Balears,1981-1990,-1.25,Negativa,27.314899,69.229856,True
3,1985-01,1985,1,Hivern,Catalunya-Balears,1981-1990,-2.87,Negativa,4.056668,55.248325,True
4,1986-02,1986,2,Hivern,Catalunya-Balears,1981-1990,-4.02,Negativa,14.940751,56.855707,True


PosixPath('../data_flourish/escena_04_nao_precipitacio_hivern.csv')

## Escena 05 — NAO i mesos amb sòl molt sec

In [8]:
target = "sol_molt_sec"

# Percentatge per fase NAO, estació i subregió
escena_05_regions = (
    df.groupby(["fase_nao", "estacio", "subregio"], observed=True)
    .agg(
        n_mesos=(target, "count"),
        n_mesos_sol_sec=(target, "sum"),
    )
    .reset_index()
)

escena_05_regions["% mesos amb sòl molt sec"] = (
    escena_05_regions["n_mesos_sol_sec"] / escena_05_regions["n_mesos"] * 100
)

escena_05_regions["Mesos sense sòl molt sec"] = (
    escena_05_regions["n_mesos"] - escena_05_regions["n_mesos_sol_sec"]
)

# Agregat global de totes les subregions, només per aquesta escena
escena_05_totes = (
    escena_05_regions
    .groupby(["fase_nao", "estacio"], observed=True)
    .agg(
        n_mesos=("n_mesos", "sum"),
        n_mesos_sol_sec=("n_mesos_sol_sec", "sum"),
    )
    .reset_index()
)

escena_05_totes["subregio"] = "Totes"

escena_05_totes["% mesos amb sòl molt sec"] = (
    escena_05_totes["n_mesos_sol_sec"] / escena_05_totes["n_mesos"] * 100
)

escena_05_totes["Mesos sense sòl molt sec"] = (
    escena_05_totes["n_mesos"] - escena_05_totes["n_mesos_sol_sec"]
)

# Unim agregat + subregions originals
escena_05_sol_sec = pd.concat(
    [escena_05_totes, escena_05_regions],
    ignore_index=True
)

# Renombrem columnes
escena_05_sol_sec = escena_05_sol_sec.rename(columns={
    "fase_nao": "Fase NAO",
    "estacio": "Estació",
    "subregio": "Subregió",
    "n_mesos": "N mesos",
    "n_mesos_sol_sec": "N mesos amb sòl molt sec",
})

escena_05_sol_sec = escena_05_sol_sec[
    [
        "Fase NAO",
        "Estació",
        "Subregió",
        "% mesos amb sòl molt sec",
        "N mesos",
        "N mesos amb sòl molt sec",
        "Mesos sense sòl molt sec",
    ]
]

# Traducció manual només per aquesta escena
escena_05_sol_sec["Fase NAO"] = escena_05_sol_sec["Fase NAO"].replace({
    "Negative": "Negativa",
    "Neutral": "Neutral",
    "Positive": "Positiva",
})

escena_05_sol_sec["Estació"] = escena_05_sol_sec["Estació"].replace({
    "Winter": "Hivern",
    "Spring": "Primavera",
    "Summer": "Estiu",
    "Autumn": "Tardor",
})

escena_05_sol_sec["Subregió"] = escena_05_sol_sec["Subregió"].replace({
    "Totes": "Totes",
    "Catalunya_Balears": "Catalunya-Balears",
    "Golf_Lleo": "Golf del Lleó",
    "Llevant_Iberic": "Llevant Ibèric",
    "Mediterrani_Central_Occidental": "Mediterrani central occidental",
})

# Ordres específics per aquesta escena
ordre_subregions_05 = [
    "Totes",
    "Catalunya-Balears",
    "Golf del Lleó",
    "Llevant Ibèric",
    "Mediterrani central occidental",
]

ordre_estacions_05 = [
    "Hivern",
    "Primavera",
    "Estiu",
    "Tardor",
]

ordre_fases_05 = [
    "Negativa",
    "Neutral",
    "Positiva",
]

escena_05_sol_sec["Subregió"] = pd.Categorical(
    escena_05_sol_sec["Subregió"],
    categories=ordre_subregions_05,
    ordered=True,
)

escena_05_sol_sec["Estació"] = pd.Categorical(
    escena_05_sol_sec["Estació"],
    categories=ordre_estacions_05,
    ordered=True,
)

escena_05_sol_sec["Fase NAO"] = pd.Categorical(
    escena_05_sol_sec["Fase NAO"],
    categories=ordre_fases_05,
    ordered=True,
)

# Ordenem perquè Flourish mostri Totes primer i les fases/estacions en ordre narratiu
escena_05_sol_sec = (
    escena_05_sol_sec
    .sort_values(["Subregió", "Estació", "Fase NAO"])
    .reset_index(drop=True)
)

# Tornem categories a text abans d'exportar
for col in ["Subregió", "Estació", "Fase NAO"]:
    escena_05_sol_sec[col] = escena_05_sol_sec[col].astype(str)

# Exportem sense traducció ni ordenació global per evitar que "Totes" es converteixi en blanc
export_csv(
    escena_05_sol_sec,
    "escena_05_nao_percent_mesos_sol_molt_sec.csv",
    traduir=False,
    ordenar=False,
)

Exportat: ../data_flourish/escena_05_nao_percent_mesos_sol_molt_sec.csv
Dimensions: (60, 7)


,Fase NAO,Estació,Subregió,% mesos amb sòl molt sec,N mesos,N mesos amb sòl molt sec,Mesos sense sòl molt sec
0,Negativa,Hivern,Totes,9.821429,112,11,101
1,Neutral,Hivern,Totes,8.695652,92,8,84
2,Positiva,Hivern,Totes,32.971014,276,91,185
3,Negativa,Primavera,Totes,20.000000,160,32,128
4,Neutral,Primavera,Totes,25.892857,112,29,83


PosixPath('../data_flourish/escena_05_nao_percent_mesos_sol_molt_sec.csv')

In [9]:
base_05_diff = escena_05_sol_sec.copy()

# Traduïm abans de pivotar perquè les fases ja surtin en català.
base_05_diff = traduir_valors_flourish(base_05_diff)

pivot_05 = base_05_diff.pivot_table(
    index=["Estació", "Subregió"],
    columns="Fase NAO",
    values="% mesos amb sòl molt sec",
    aggfunc="mean",
    observed=True,
).reset_index()

for col in ["Negativa", "Neutral", "Positiva"]:
    if col not in pivot_05.columns:
        pivot_05[col] = np.nan

escena_05_diff = pivot_05.copy()
escena_05_diff["Diferència positiva - negativa"] = (
    escena_05_diff["Positiva"] - escena_05_diff["Negativa"]
)

escena_05_diff = escena_05_diff.rename(columns={
    "Negativa": "% mesos sòl molt sec fase negativa",
    "Neutral": "% mesos sòl molt sec fase neutral",
    "Positiva": "% mesos sòl molt sec fase positiva",
})

escena_05_diff = escena_05_diff[
    [
        "Estació",
        "Subregió",
        "% mesos sòl molt sec fase negativa",
        "% mesos sòl molt sec fase neutral",
        "% mesos sòl molt sec fase positiva",
        "Diferència positiva - negativa",
    ]
]

export_csv(
    escena_05_diff,
    "escena_05_nao_diferencia_sol_molt_sec.csv",
    traduir=False
)

Exportat: ../data_flourish/escena_05_nao_diferencia_sol_molt_sec.csv
Dimensions: (20, 6)


/tmp/ipykernel_543/2624149760.py:102: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  out["Subregió"] = pd.Categorical(out["Subregió"], categories=ordre_subregions_cat, ordered=True)


Fase NAO,Estació,Subregió,% mesos sòl molt sec fase negativa,% mesos sòl molt sec fase neutral,% mesos sòl molt sec fase positiva,Diferència positiva - negativa
0,Hivern,Catalunya-Balears,7.142857,4.347826,31.884058,24.741201
1,Hivern,Golf del Lleó,3.571429,4.347826,15.942029,12.370600
2,Hivern,Llevant Ibèric,21.428571,21.739130,49.275362,27.846791
3,Hivern,Mediterrani central occidental,7.142857,4.347826,34.782609,27.639752
4,Hivern,NaN,9.821429,8.695652,32.971014,23.149586


PosixPath('../data_flourish/escena_05_nao_diferencia_sol_molt_sec.csv')

## Escena 06 — WeMO, temperatura i SST

In [10]:
# Calculem correlacions

CORR_EXHAUSTIVES = PROC / "04_correlacions_exhaustives.csv"

if CORR_EXHAUSTIVES.exists():
    corr_exhaustives = pd.read_csv(CORR_EXHAUSTIVES)
    print("Carregat:", CORR_EXHAUSTIVES)
else:
    print("No existeix 04_correlacions_exhaustives.csv. Es calculen correlacions bàsiques per estació.")

    rows = []
    for estacio, subset in df.groupby("estacio", observed=True):
        for tele in ["nao", "wemo", "roni"]:
            for var in ordre_variables_narratives:
                tmp = subset[[tele, var]].dropna()
                corr = tmp[tele].corr(tmp[var]) if len(tmp) >= 20 else np.nan

                rows.append({
                    "ambit": "estacio",
                    "estacio": estacio,
                    "teleconnexio": tele,
                    "variable": var,
                    "n": len(tmp),
                    "pearson_r": corr,
                    "pearson_p": np.nan,
                    "pearson_q_fdr": np.nan,
                })

    corr_exhaustives = pd.DataFrame(rows)

print(corr_exhaustives.shape)
display(corr_exhaustives.head())

Carregat: ../data_processed/04_correlacions_exhaustives.csv
(6000, 15)


,ambit,teleconnexio,variable,n,pearson_r,pearson_p,spearman_r,spearman_p,abs_pearson_r,abs_spearman_r,subregio,estacio,periode,pearson_q_fdr,spearman_q_fdr
0,subregio_estacio_periode,wemo,vent_u10,30,0.938967,1.674279e-14,0.924669,2.915346e-13,0.938967,0.924669,Mediterrani_Central_Occidental,Winter,2011-2020,7.386524e-13,1.286182e-11
1,subregio_estacio_periode,wemo,vent_u10,30,0.927687,1.676180e-13,0.936033,3.172028e-14,0.927687,0.936033,Mediterrani_Central_Occidental,Winter,1991-2000,6.841551e-12,1.560014e-12
2,subregio_estacio_periode,wemo,vent_u10,30,0.914127,1.707096e-12,0.857937,1.371268e-09,0.914127,0.857937,Golf_Lleo,Winter,1991-2000,6.322577e-11,3.722900e-08
3,subregio_estacio_periode,wemo,vent_u10,30,0.904896,6.725161e-12,0.855458,1.719325e-09,0.904896,0.855458,Catalunya_Balears,Winter,2011-2020,2.332426e-10,4.646826e-08
4,subregio_estacio_periode,wemo,vent_u10,30,0.904276,7.337154e-12,0.874082,2.810724e-10,0.904276,0.874082,Catalunya_Balears,Winter,1981-1990,2.515596e-10,9.018365e-09


In [11]:
variables_escena_06 = {
    "anomalia_temperatura": {
        "Variable": "Temperatura",
        "Unitat": "°C",
    },
    "anomalia_sst": {
        "Variable": "SST",
        "Unitat": "°C",
    },
}

rows = []

for var_col, meta in variables_escena_06.items():
    grouped = (
        df.groupby(["estacio", "subregio", "fase_wemo"], observed=True)
        .agg(
            anomalia_mitjana=(var_col, "mean"),
            n_mesos=(var_col, "count"),
        )
        .reset_index()
    )
    
    grouped["Variable"] = meta["Variable"]
    grouped["Unitat"] = meta["Unitat"]
    
    rows.append(grouped)
    
escena_06_fases = pd.concat(rows, ignore_index=True)

escena_06_fases = escena_06_fases.rename(columns={
    "estacio": "Estació",
    "subregio": "Subregió",
    "fase_wemo": "Fase WeMO",
    "anomalia_mitjana": "Anomalia mitjana",
    "n_mesos": "N mesos",
})

escena_06_fases = escena_06_fases[
    [
        "Estació",
        "Subregió",
        "Fase WeMO",
        "Variable",
        "Unitat",
        "Anomalia mitjana",
        "N mesos",
    ]
]

In [12]:
rows = []

for var_col, meta in variables_escena_06.items():
    grouped = (
        df.groupby(["estacio", "fase_wemo"], observed=True)
        .agg(
            anomalia_mitjana=(var_col, "mean"),
            n_mesos=(var_col, "count"),
        )
        .reset_index()
    )

    grouped["Variable"] = meta["Variable"]
    grouped["Unitat"] = meta["Unitat"]

    rows.append(grouped)

escena_06_global = pd.concat(rows, ignore_index=True)

escena_06_global = escena_06_global[
    escena_06_global["fase_wemo"].isin(["Negative", "Positive"])
].copy()

escena_06_global = escena_06_global.rename(columns={
    "estacio": "Estació",
    "fase_wemo": "Fase WeMO",
    "anomalia_mitjana": "Anomalia mitjana",
    "n_mesos": "N mesos",
})

escena_06_global = escena_06_global[
    [
        "Estació",
        "Fase WeMO",
        "Variable",
        "Unitat",
        "Anomalia mitjana",
        "N mesos",
    ]
]

export_csv(
    escena_06_global,
    "escena_06_wemo_fases_temperatura_sst_global.csv"
)

Exportat: ../data_flourish/escena_06_wemo_fases_temperatura_sst_global.csv
Dimensions: (16, 6)


,Estació,Fase WeMO,Variable,Unitat,Anomalia mitjana,N mesos
0,Hivern,Negativa,SST,°C,0.027385,132
1,Hivern,Negativa,Temperatura,°C,0.116845,132
2,Primavera,Negativa,SST,°C,0.168883,192
3,Primavera,Negativa,Temperatura,°C,0.279386,192
4,Estiu,Negativa,SST,°C,0.236191,172


PosixPath('../data_flourish/escena_06_wemo_fases_temperatura_sst_global.csv')

## Escena 07 — Comparació entre NAO, WeMO i RONI

In [13]:
# Escena 07 — Distribució de la força de les correlacions
# Objectiu: mostrar que RONI/ENSO té un senyal més feble i menys consistent
# que NAO i WeMO en les variables narratives principals.

# Ens assegurem que tenim carregades les correlacions exhaustives del Notebook 04.
CORR_EXHAUSTIVES = PROC / "04_correlacions_exhaustives.csv"

if CORR_EXHAUSTIVES.exists():
    corr_exhaustives = pd.read_csv(CORR_EXHAUSTIVES)
    print("Carregat:", CORR_EXHAUSTIVES)
else:
    raise FileNotFoundError(
        "No s'ha trobat 04_correlacions_exhaustives.csv. "
        "Executa primer el Notebook 04 o copia aquest fitxer a data_processed."
    )

# Variables narratives que volem mostrar a la web.
variables_escena_07 = {
    "anomalia_precipitacio_mensual_mm": "Precipitació",
    "anomalia_humitat_sol_capa1": "Humitat del sòl",
    "anomalia_temperatura": "Temperatura",
    "anomalia_sst": "SST",
}

unitats_escena_07 = {
    "anomalia_precipitacio_mensual_mm": "mm",
    "anomalia_humitat_sol_capa1": "m3/m3",
    "anomalia_temperatura": "°C",
    "anomalia_sst": "°C",
}

# Àmbits que mantenim per no sobrecarregar el gràfic.
ambits_escena_07 = [
    "global",
    "estacio",
    "subregio",
    "subregio_estacio",
]

# Filtrem només les relacions que volem ensenyar.
escena_07_dist = corr_exhaustives[
    corr_exhaustives["teleconnexio"].isin(["nao", "wemo", "roni"])
    & corr_exhaustives["variable"].isin(variables_escena_07.keys())
    & corr_exhaustives["ambit"].isin(ambits_escena_07)
].copy()

# Columnes opcionals: segons el fitxer poden existir o no.
for col in ["subregio", "estacio", "periode", "pearson_p", "pearson_q_fdr", "spearman_r", "spearman_p"]:
    if col not in escena_07_dist.columns:
        escena_07_dist[col] = np.nan

# Traducció de teleconnexions.
escena_07_dist["Teleconnexió"] = escena_07_dist["teleconnexio"].map({
    "nao": "NAO",
    "wemo": "WeMO",
    "roni": "RONI",
})

# Traducció de variables.
escena_07_dist["Variable"] = escena_07_dist["variable"].map(variables_escena_07)
escena_07_dist["Unitat"] = escena_07_dist["variable"].map(unitats_escena_07)

# Traducció d'àmbits.
escena_07_dist["Àmbit"] = escena_07_dist["ambit"].replace({
    "global": "Global",
    "estacio": "Estació",
    "subregio": "Subregió",
    "subregio_estacio": "Subregió i estació",
})

# Traducció d'estacions i subregions.
escena_07_dist["Estació"] = escena_07_dist["estacio"].replace({
    "Winter": "Hivern",
    "Spring": "Primavera",
    "Summer": "Estiu",
    "Autumn": "Tardor",
})

escena_07_dist["Subregió"] = escena_07_dist["subregio"].replace({
    "Catalunya_Balears": "Catalunya-Balears",
    "Golf_Lleo": "Golf del Lleó",
    "Llevant_Iberic": "Llevant Ibèric",
    "Mediterrani_Central_Occidental": "Mediterrani central occidental",
})

# Per als àmbits globals o només estacionals, no hi ha subregió.
# Posem "Totes" perquè el tooltip i els filtres siguin més clars.
escena_07_dist["Subregió"] = escena_07_dist["Subregió"].fillna("Totes")

# Per als àmbits globals o només regionals, no hi ha estació.
# Posem "Totes" pel mateix motiu.
escena_07_dist["Estació"] = escena_07_dist["Estació"].fillna("Totes")

# Valors principals.
escena_07_dist["Correlació"] = escena_07_dist["pearson_r"]
escena_07_dist["Força absoluta"] = escena_07_dist["Correlació"].abs()
escena_07_dist["P valor"] = escena_07_dist["pearson_p"]
escena_07_dist["Q FDR"] = escena_07_dist["pearson_q_fdr"]

# Significació FDR.
# Si no hi ha q-value, marquem "No disponible".
escena_07_dist["Significatiu FDR"] = np.where(
    escena_07_dist["Q FDR"].notna(),
    np.where(escena_07_dist["Q FDR"] < 0.05, "Sí", "No"),
    "No disponible",
)

# Signe de la correlació per si vols filtrar o posar al tooltip.
escena_07_dist["Signe"] = np.where(
    escena_07_dist["Correlació"] > 0,
    "Positiva",
    np.where(escena_07_dist["Correlació"] < 0, "Negativa", "Zero")
)

# Etiqueta curta per tooltip.
escena_07_dist["Etiqueta"] = (
    escena_07_dist["Teleconnexió"]
    + " · "
    + escena_07_dist["Variable"]
    + " · "
    + escena_07_dist["Àmbit"]
)

# Arrodoniments per fer els tooltips més nets.
escena_07_dist["Correlació"] = escena_07_dist["Correlació"].round(3)
escena_07_dist["Força absoluta"] = escena_07_dist["Força absoluta"].round(3)
escena_07_dist["P valor"] = escena_07_dist["P valor"].round(4)
escena_07_dist["Q FDR"] = escena_07_dist["Q FDR"].round(4)

# Ordenació narrativa.
ordre_teleconnexions_07 = ["NAO", "WeMO", "RONI"]
ordre_variables_07 = ["Precipitació", "Humitat del sòl", "Temperatura", "SST"]
ordre_ambits_07 = ["Global", "Estació", "Subregió", "Subregió i estació"]
ordre_estacions_07 = ["Totes", "Hivern", "Primavera", "Estiu", "Tardor"]
ordre_subregions_07 = [
    "Totes",
    "Catalunya-Balears",
    "Golf del Lleó",
    "Llevant Ibèric",
    "Mediterrani central occidental",
]

escena_07_dist["Teleconnexió"] = pd.Categorical(
    escena_07_dist["Teleconnexió"],
    categories=ordre_teleconnexions_07,
    ordered=True,
)

escena_07_dist["Variable"] = pd.Categorical(
    escena_07_dist["Variable"],
    categories=ordre_variables_07,
    ordered=True,
)

escena_07_dist["Àmbit"] = pd.Categorical(
    escena_07_dist["Àmbit"],
    categories=ordre_ambits_07,
    ordered=True,
)

escena_07_dist["Estació"] = pd.Categorical(
    escena_07_dist["Estació"],
    categories=ordre_estacions_07,
    ordered=True,
)

escena_07_dist["Subregió"] = pd.Categorical(
    escena_07_dist["Subregió"],
    categories=ordre_subregions_07,
    ordered=True,
)

escena_07_dist = (
    escena_07_dist
    .sort_values([
        "Teleconnexió",
        "Àmbit",
        "Variable",
        "Estació",
        "Subregió",
        "Força absoluta",
    ])
    .reset_index(drop=True)
)

# Tornem categories a text abans d'exportar.
for col in ["Teleconnexió", "Variable", "Àmbit", "Estació", "Subregió"]:
    escena_07_dist[col] = escena_07_dist[col].astype(str)

# Deixem només les columnes que necessitem per Flourish.
escena_07_dist = escena_07_dist[
    [
        "Teleconnexió",
        "Variable",
        "Unitat",
        "Àmbit",
        "Subregió",
        "Estació",
        "Correlació",
        "Força absoluta",
        "Signe",
        "P valor",
        "Q FDR",
        "Significatiu FDR",
        "n",
        "Etiqueta",
    ]
].rename(columns={
    "n": "N",
})

# Comprovació ràpida
print("Dimensions escena 07 distribució:", escena_07_dist.shape)
display(escena_07_dist.head())
display(
    escena_07_dist
    .groupby("Teleconnexió", observed=True)
    .agg(
        n_relacions=("Força absoluta", "count"),
        força_mitjana=("Força absoluta", "mean"),
        força_mediana=("Força absoluta", "median"),
        força_maxima=("Força absoluta", "max"),
    )
    .reset_index()
)

# Exportació
export_csv(
    escena_07_dist,
    "escena_07_distribucio_forca_correlacions.csv",
    traduir=False,
    ordenar=False,
)

Carregat: ../data_processed/04_correlacions_exhaustives.csv
Dimensions escena 07 distribució: (300, 14)


,Teleconnexió,Variable,Unitat,Àmbit,Subregió,Estació,Correlació,Força absoluta,Signe,P valor,Q FDR,Significatiu FDR,N,Etiqueta
0,NAO,Precipitació,mm,Global,Totes,Totes,-0.312,0.312,Negativa,0.0,0.0,Sí,1920,NAO · Precipitació · Global
1,NAO,Humitat del sòl,m3/m3,Global,Totes,Totes,-0.223,0.223,Negativa,0.0,0.0,Sí,1920,NAO · Humitat del sòl · Global
2,NAO,Temperatura,°C,Global,Totes,Totes,0.195,0.195,Positiva,0.0,0.0,Sí,1920,NAO · Temperatura · Global
3,NAO,SST,°C,Global,Totes,Totes,0.114,0.114,Positiva,0.0,0.0,Sí,1920,NAO · SST · Global
4,NAO,Precipitació,mm,Estació,Totes,Hivern,-0.456,0.456,Negativa,0.0,0.0,Sí,480,NAO · Precipitació · Estació


,Teleconnexió,n_relacions,força_mitjana,força_mediana,força_maxima
0,NAO,100,0.22942,0.2270,0.555
1,RONI,100,0.09585,0.0765,0.254
2,WeMO,100,0.21567,0.2075,0.508


Exportat: ../data_flourish/escena_07_distribucio_forca_correlacions.csv
Dimensions: (300, 14)


,Teleconnexió,Variable,Unitat,Àmbit,Subregió,Estació,Correlació,Força absoluta,Signe,P valor,Q FDR,Significatiu FDR,N,Etiqueta
0,NAO,Precipitació,mm,Global,Totes,Totes,-0.312,0.312,Negativa,0.0,0.0,Sí,1920,NAO · Precipitació · Global
1,NAO,Humitat del sòl,m3/m3,Global,Totes,Totes,-0.223,0.223,Negativa,0.0,0.0,Sí,1920,NAO · Humitat del sòl · Global
2,NAO,Temperatura,°C,Global,Totes,Totes,0.195,0.195,Positiva,0.0,0.0,Sí,1920,NAO · Temperatura · Global
3,NAO,SST,°C,Global,Totes,Totes,0.114,0.114,Positiva,0.0,0.0,Sí,1920,NAO · SST · Global
4,NAO,Precipitació,mm,Estació,Totes,Hivern,-0.456,0.456,Negativa,0.0,0.0,Sí,480,NAO · Precipitació · Estació


PosixPath('../data_flourish/escena_07_distribucio_forca_correlacions.csv')

## Escena 08 — Episodis càlids

In [14]:
target = "episodi_calid_i_sol_sec"

escena_08 = (
    df.groupby(["subregio", "periode"], observed=True)
    .agg(
        n_mesos=(target, "count"),
        n_episodis=(target, "sum"),
    )
    .reset_index()
)

escena_08["Mesos sense episodi"] = escena_08["n_mesos"] - escena_08["n_episodis"]
escena_08["% mesos càlids amb baixa humitat del sòl"] = (
    escena_08["n_episodis"] / escena_08["n_mesos"] * 100
)

escena_08 = escena_08.rename(columns={
    "subregio": "Subregió",
    "periode": "Període",
    "n_mesos": "N mesos",
    "n_episodis": "N episodis",
})

escena_08 = escena_08[
    [
        "Subregió",
        "Període",
        "% mesos càlids amb baixa humitat del sòl",
        "N mesos",
        "N episodis",
        "Mesos sense episodi",
    ]
]

In [15]:
base_08_wide = traduir_valors_flourish(escena_08)

escena_08_wide = base_08_wide.pivot_table(
    index="Període",
    columns="Subregió",
    values="% mesos càlids amb baixa humitat del sòl",
    aggfunc="mean",
    observed=True,
).reset_index()

# Ordenem les files pel període
ordre_periodes = ["1981-1990", "1991-2000", "2001-2010", "2011-2020"]

escena_08_wide["Període"] = pd.Categorical(
    escena_08_wide["Període"],
    categories=ordre_periodes,
    ordered=True
)

escena_08_wide = (
    escena_08_wide
    .sort_values("Període")
    .reset_index(drop=True)
)

# Ordenem les columnes de subregió
cols_subregions = [
    "Catalunya-Balears",
    "Golf del Lleó",
    "Llevant Ibèric",
    "Mediterrani central occidental",
]

escena_08_wide = escena_08_wide[
    ["Període"] + [c for c in cols_subregions if c in escena_08_wide.columns]
]

# Tornem Període a text abans d'exportar
escena_08_wide["Període"] = escena_08_wide["Període"].astype(str)

export_csv(
    escena_08_wide,
    "escena_08_episodis_calor_sol_sec_wide_subregions.csv",
    traduir=False
)

Exportat: ../data_flourish/escena_08_episodis_calor_sol_sec_wide_subregions.csv
Dimensions: (4, 5)


Subregió,Període,Catalunya-Balears,Golf del Lleó,Llevant Ibèric,Mediterrani central occidental
0,1981-1990,0.833333,3.333333,2.500000,2.500000
1,1991-2000,4.166667,2.500000,4.166667,2.500000
2,2001-2010,6.666667,6.666667,10.000000,4.166667
3,2011-2020,11.666667,14.166667,12.500000,10.833333


PosixPath('../data_flourish/escena_08_episodis_calor_sol_sec_wide_subregions.csv')

## Escena 09 — Explorador lliure

Aquesta escena genera dos CSV pensats per alimentar una zona d'exploració lliure a la web.

El primer CSV resumeix anomalies mitjanes per teleconnexió, fase, variable, subregió i estació.

El segon CSV resumeix la freqüència d'episodis o condicions extremes per teleconnexió, fase, subregió i estació.

In [16]:
# Configuració específica de l'escena 09

# Els CSV de l'escena 09 es guarden amb la resta de CSV de Flourish
print("FLOURISH:", FLOURISH.resolve())

# Traduccions de valors categòrics
trad_estacions_09 = {
    "Winter": "Hivern",
    "Spring": "Primavera",
    "Summer": "Estiu",
    "Autumn": "Tardor",
}

trad_fases_09 = {
    "Negative": "Negativa",
    "Neutral": "Neutral",
    "Positive": "Positiva",
}

# Per a l'escena 09 volem que RONI/ENSO també funcioni amb fases negativa/neutral/positiva.
# Nota metodològica: Negativa = La Niña, Positiva = El Niño.
trad_enso_09 = {
    "La Nina": "Negativa",
    "Neutral": "Neutral",
    "El Nino": "Positiva",
}

trad_subregions_09 = {
    "Llevant_Iberic": "Llevant Ibèric",
    "Catalunya_Balears": "Catalunya-Balears",
    "Golf_Lleo": "Golf del Lleó",
    "Mediterrani_Central_Occidental": "Mediterrani central occidental",
}

# Ordres per deixar els CSV preparats per la web
ordre_teleconnexions_09 = [
    "NAO",
    "WeMO",
    "RONI/ENSO",
]

ordre_fases_generals_09 = [
    "Negativa",
    "Neutral",
    "Positiva",
]

ordre_variables_09 = [
    "Precipitació",
    "Temperatura",
    "Humitat del sòl",
    "SST",
]

ordre_estacions_09 = [
    "Hivern",
    "Primavera",
    "Estiu",
    "Tardor",
]

ordre_subregions_09 = [
    "Llevant Ibèric",
    "Catalunya-Balears",
    "Golf del Lleó",
    "Mediterrani central occidental",
]

ordre_periodes_09 = [
    "Tots",
    "1981-1990",
    "1991-2000",
    "2001-2010",
    "2011-2020",
]

FLOURISH: /workspace/data_flourish


In [17]:
# Variables climàtiques que entren a l'explorador d'anomalies
variables_explorador_09 = {
    "anomalia_precipitacio_mensual_mm": {
        "Variable": "Precipitació",
        "Unitat": "mm",
    },
    "anomalia_temperatura": {
        "Variable": "Temperatura",
        "Unitat": "°C",
    },
    "anomalia_humitat_sol_capa1": {
        "Variable": "Humitat del sòl",
        "Unitat": "m3/m3",
    },
    "anomalia_sst": {
        "Variable": "SST",
        "Unitat": "°C",
    },
}

# Teleconnexions i columna de fase corresponent
teleconnexions_explorador_09 = {
    "NAO": {
        "fase_col": "fase_nao",
        "trad_fases": trad_fases_09,
    },
    "WeMO": {
        "fase_col": "fase_wemo",
        "trad_fases": trad_fases_09,
    },
    "RONI/ENSO": {
        "fase_col": "fase_enso",
        "trad_fases": trad_enso_09,
    },
}

# Indicadors per al segon CSV
indicadors_episodis_09 = {
    "episodi_calid_i_sol_sec": "Episodi càlid amb baixa humitat del sòl",
    "episodi_calid_i_sec": "Episodi càlid i sec",
    "episodi_fred_i_humit": "Episodi fred i humit",
    "sol_molt_sec": "Sòl molt sec",
    "sol_molt_humit": "Sòl molt humit",
    "mes_molt_calid": "Mes molt càlid",
    "mes_molt_fred": "Mes molt fred",
    "mes_molt_sec": "Mes molt sec",
    "mes_molt_humit": "Mes molt humit",
}

# Comprovació de columnes necessàries
cols_necessaries_09 = [
    "estacio",
    "subregio",
    "periode",
    "fase_nao",
    "fase_wemo",
    "fase_enso",
] + list(variables_explorador_09.keys()) + list(indicadors_episodis_09.keys())

missing_09 = [c for c in cols_necessaries_09 if c not in df.columns]

if missing_09:
    raise ValueError(f"Falten columnes necessàries per a l'escena 09: {missing_09}")

print("Columnes necessàries disponibles.")

Columnes necessàries disponibles.


In [18]:
def ordenar_escena_09(dataframe):
    """
    Ordena les files dels CSV de l'escena 09 segons l'ordre narratiu previst.
    """
    out = dataframe.copy()

    if "Teleconnexió" in out.columns:
        out["Teleconnexió"] = pd.Categorical(
            out["Teleconnexió"],
            categories=ordre_teleconnexions_09,
            ordered=True,
        )

    if "Fase" in out.columns:
        out["Fase"] = pd.Categorical(
            out["Fase"],
            categories=ordre_fases_generals_09,
            ordered=True,
        )

    if "Variable" in out.columns:
        out["Variable"] = pd.Categorical(
            out["Variable"],
            categories=ordre_variables_09,
            ordered=True,
        )

    if "Indicador" in out.columns:
        out["Indicador"] = pd.Categorical(
            out["Indicador"],
            categories=list(indicadors_episodis_09.values()),
            ordered=True,
        )

    if "Estació" in out.columns:
        out["Estació"] = pd.Categorical(
            out["Estació"],
            categories=ordre_estacions_09,
            ordered=True,
        )

    if "Subregió" in out.columns:
        out["Subregió"] = pd.Categorical(
            out["Subregió"],
            categories=ordre_subregions_09,
            ordered=True,
        )

    if "Període" in out.columns:
        out["Període"] = pd.Categorical(
            out["Període"],
            categories=ordre_periodes_09,
            ordered=True,
        )

    sort_cols = [
        c for c in [
            "Teleconnexió",
            "Fase",
            "Variable",
            "Indicador",
            "Subregió",
            "Estació",
            "Període",
        ]
        if c in out.columns
    ]

    out = out.sort_values(sort_cols).reset_index(drop=True)

    # Tornem categories a text abans d'exportar
    for col in out.select_dtypes(include="category").columns:
        out[col] = out[col].astype(str)

    return out


def export_escena09_csv(dataframe, filename):
    """
    Exporta un CSV de l'escena 09 a la mateixa carpeta que la resta de CSV de Flourish.
    """
    path = FLOURISH / filename
    dataframe.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Exportat: {path}")
    print("Dimensions:", dataframe.shape)
    display(dataframe.head())
    return path

In [19]:
rows = []

for teleconnexio, tele_meta in teleconnexions_explorador_09.items():
    fase_col = tele_meta["fase_col"]
    trad_fases = tele_meta["trad_fases"]

    for var_col, var_meta in variables_explorador_09.items():

        # ------------------------------------------------------------
        # 1) Agregat total: Període = Tots
        # ------------------------------------------------------------
        grouped_tots = (
            df.groupby([fase_col, "subregio", "estacio"], observed=True)
            .agg(
                anomalia_mitjana=(var_col, "mean"),
                n_mesos=(var_col, "count"),
            )
            .reset_index()
        )

        grouped_tots["Període"] = "Tots"

        # ------------------------------------------------------------
        # 2) Agregat per dècada: Període = periode
        # ------------------------------------------------------------
        grouped_periode = (
            df.groupby([fase_col, "subregio", "estacio", "periode"], observed=True)
            .agg(
                anomalia_mitjana=(var_col, "mean"),
                n_mesos=(var_col, "count"),
            )
            .reset_index()
            .rename(columns={"periode": "Període"})
        )

        # ------------------------------------------------------------
        # 3) Unim Tots + dècades
        # ------------------------------------------------------------
        grouped = pd.concat(
            [grouped_tots, grouped_periode],
            ignore_index=True,
        )

        grouped["Teleconnexió"] = teleconnexio
        grouped["Fase"] = grouped[fase_col].replace(trad_fases)
        grouped["Variable"] = var_meta["Variable"]
        grouped["Unitat"] = var_meta["Unitat"]
        grouped["Subregió"] = grouped["subregio"].replace(trad_subregions_09)
        grouped["Estació"] = grouped["estacio"].replace(trad_estacions_09)

        grouped = grouped.rename(columns={
            "anomalia_mitjana": "Anomalia mitjana",
            "n_mesos": "N mesos",
        })

        grouped = grouped[
            [
                "Teleconnexió",
                "Fase",
                "Variable",
                "Subregió",
                "Estació",
                "Període",
                "Anomalia mitjana",
                "Unitat",
                "N mesos",
            ]
        ]

        rows.append(grouped)

escena_09_anomalies = pd.concat(rows, ignore_index=True)

# Arrodonim per fer el CSV més net
escena_09_anomalies["Anomalia mitjana"] = escena_09_anomalies["Anomalia mitjana"].round(4)

# Ordenem
escena_09_anomalies = ordenar_escena_09(escena_09_anomalies)

# Comprovació: una fila per combinació
duplicats_anomalies_09 = escena_09_anomalies.duplicated(
    subset=["Teleconnexió", "Fase", "Variable", "Subregió", "Estació", "Període"],
    keep=False,
)

print("Duplicats escena 09 anomalies:", duplicats_anomalies_09.sum())

if duplicats_anomalies_09.sum() > 0:
    display(
        escena_09_anomalies[duplicats_anomalies_09]
        .sort_values([
            "Teleconnexió",
            "Fase",
            "Variable",
            "Subregió",
            "Estació",
            "Període",
        ])
    )

# Comprovació ràpida de períodes
display(
    escena_09_anomalies["Període"]
    .value_counts()
    .reindex(ordre_periodes_09)
)

export_escena09_csv(
    escena_09_anomalies,
    "escena_09_explorador_anomalies.csv",
)

Duplicats escena 09 anomalies: 0


Període
Tots         576
1981-1990    576
1991-2000    576
2001-2010    576
2011-2020    576
Name: count, dtype: int64

Exportat: ../data_flourish/escena_09_explorador_anomalies.csv
Dimensions: (2880, 9)


,Teleconnexió,Fase,Variable,Subregió,Estació,Període,Anomalia mitjana,Unitat,N mesos
0,NAO,Negativa,Precipitació,Llevant Ibèric,Hivern,Tots,9.7824,mm,28
1,NAO,Negativa,Precipitació,Llevant Ibèric,Hivern,1981-1990,10.5470,mm,8
2,NAO,Negativa,Precipitació,Llevant Ibèric,Hivern,1991-2000,19.9703,mm,5
3,NAO,Negativa,Precipitació,Llevant Ibèric,Hivern,2001-2010,8.3557,mm,13
4,NAO,Negativa,Precipitació,Llevant Ibèric,Hivern,2011-2020,-9.4719,mm,2


PosixPath('../data_flourish/escena_09_explorador_anomalies.csv')

In [20]:
rows = []

for teleconnexio, tele_meta in teleconnexions_explorador_09.items():
    fase_col = tele_meta["fase_col"]
    trad_fases = tele_meta["trad_fases"]

    for indicador_col, indicador_label in indicadors_episodis_09.items():

        temp = df.copy()
        temp[indicador_col] = temp[indicador_col].astype(bool)

        # ------------------------------------------------------------
        # 1) Agregat total: Període = Tots
        # ------------------------------------------------------------
        grouped_tots = (
            temp.groupby([fase_col, "estacio", "subregio"], observed=True)
            .agg(
                n_mesos=(indicador_col, "count"),
                n_episodis=(indicador_col, "sum"),
            )
            .reset_index()
        )

        grouped_tots["Període"] = "Tots"

        # ------------------------------------------------------------
        # 2) Agregat per dècada: Període = periode
        # ------------------------------------------------------------
        grouped_periode = (
            temp.groupby([fase_col, "estacio", "subregio", "periode"], observed=True)
            .agg(
                n_mesos=(indicador_col, "count"),
                n_episodis=(indicador_col, "sum"),
            )
            .reset_index()
            .rename(columns={"periode": "Període"})
        )

        # ------------------------------------------------------------
        # 3) Unim Tots + dècades
        # ------------------------------------------------------------
        grouped = pd.concat(
            [grouped_tots, grouped_periode],
            ignore_index=True,
        )

        grouped["Percentatge mesos"] = (
            grouped["n_episodis"] / grouped["n_mesos"] * 100
        )

        grouped["Indicador"] = indicador_label
        grouped["Teleconnexió"] = teleconnexio
        grouped["Fase"] = grouped[fase_col].replace(trad_fases)
        grouped["Estació"] = grouped["estacio"].replace(trad_estacions_09)
        grouped["Subregió"] = grouped["subregio"].replace(trad_subregions_09)

        grouped = grouped.rename(columns={
            "n_episodis": "N episodis",
            "n_mesos": "N mesos",
        })

        grouped = grouped[
            [
                "Indicador",
                "Teleconnexió",
                "Fase",
                "Estació",
                "Subregió",
                "Període",
                "Percentatge mesos",
                "N episodis",
                "N mesos",
            ]
        ]

        rows.append(grouped)

escena_09_episodis = pd.concat(rows, ignore_index=True)

# Arrodonim percentatge
escena_09_episodis["Percentatge mesos"] = escena_09_episodis["Percentatge mesos"].round(2)

# Ordenem
escena_09_episodis = ordenar_escena_09(escena_09_episodis)

# Comprovació: una fila per combinació
duplicats_episodis_09 = escena_09_episodis.duplicated(
    subset=[
        "Indicador",
        "Teleconnexió",
        "Fase",
        "Estació",
        "Subregió",
        "Període",
    ],
    keep=False,
)

print("Duplicats escena 09 episodis:", duplicats_episodis_09.sum())

if duplicats_episodis_09.sum() > 0:
    display(
        escena_09_episodis[duplicats_episodis_09]
        .sort_values([
            "Indicador",
            "Teleconnexió",
            "Fase",
            "Estació",
            "Subregió",
            "Període",
        ])
    )

# Comprovació ràpida de períodes
display(
    escena_09_episodis["Període"]
    .value_counts()
    .reindex(ordre_periodes_09)
)

export_escena09_csv(
    escena_09_episodis,
    "escena_09_explorador_episodis.csv",
)

Duplicats escena 09 episodis: 0


Període
Tots         1296
1981-1990    1296
1991-2000    1296
2001-2010    1296
2011-2020    1296
Name: count, dtype: int64

Exportat: ../data_flourish/escena_09_explorador_episodis.csv
Dimensions: (6480, 9)


,Indicador,Teleconnexió,Fase,Estació,Subregió,Període,Percentatge mesos,N episodis,N mesos
0,Episodi càlid amb baixa humitat del sòl,NAO,Negativa,Hivern,Llevant Ibèric,Tots,3.57,1,28
1,Episodi càlid amb baixa humitat del sòl,NAO,Negativa,Hivern,Llevant Ibèric,1981-1990,12.50,1,8
2,Episodi càlid amb baixa humitat del sòl,NAO,Negativa,Hivern,Llevant Ibèric,1991-2000,0.00,0,5
3,Episodi càlid amb baixa humitat del sòl,NAO,Negativa,Hivern,Llevant Ibèric,2001-2010,0.00,0,13
4,Episodi càlid amb baixa humitat del sòl,NAO,Negativa,Hivern,Llevant Ibèric,2011-2020,0.00,0,2


PosixPath('../data_flourish/escena_09_explorador_episodis.csv')

In [21]:
fitxers_escena_09 = [
    FLOURISH / "escena_09_explorador_anomalies.csv",
    FLOURISH / "escena_09_explorador_episodis.csv",
]

resum_09 = []

for path in fitxers_escena_09:
    temp = pd.read_csv(path)
    resum_09.append({
        "Fitxer": path.name,
        "Files": temp.shape[0],
        "Columnes": temp.shape[1],
        "Columnes disponibles": ", ".join(temp.columns),
    })

resum_09 = pd.DataFrame(resum_09)
display(resum_09)

,Fitxer,Files,Columnes,Columnes disponibles
0,escena_09_explorador_anomalies.csv,2880,9,"Teleconnexió, Fase, Variable, Subregió, Estaci..."
1,escena_09_explorador_episodis.csv,6480,9,"Indicador, Teleconnexió, Fase, Estació, Subreg..."
